# Exploração do dataset — Give Me Some Credit

**Objetivo:** EDA estruturada para descobrir problemas reais antes de escrever código de produção.  
As regras e thresholds que derivamos aqui alimentam diretamente `validate_dataframe()` e `merge_sources()` em `src/ingestion.py`.

Roteiro:
1. Schema e tipos
2. Ausência de dados (nulos)
3. Taxa de inadimplência
4. Violações de domínio
5. Distribuição de renda (justifica escolha de mediana na imputação)

In [ ]:
import pandas as pd

df = pd.read_csv('data/raw/cs-training.csv', index_col=0)
print('Shape:', df.shape)
df.info()

## 1. Schema e tipos

**Hipótese:** O dataset tem 11 colunas após `index_col=0`. `SeriousDlqin2yrs` é inteiro (target binário) e `MonthlyIncome` é float64 (aceita nulos).

In [ ]:
print('Colunas:', df.columns.tolist())
print('\nTipos:')
print(df.dtypes)

## 2. Ausência de dados

**Hipótese:** Apenas `MonthlyIncome` e `NumberOfDependents` têm nulos — as demais colunas são completas.
> Resultado esperado: ~20% em `MonthlyIncome`, ~3% em `NumberOfDependents`.

In [ ]:
null_pct = (df.isnull().sum() / len(df) * 100).round(2)
print('Nulos por coluna (%):')
print(null_pct[null_pct > 0])

## 3. Taxa de inadimplência

**Hipótese:** Dataset desbalanceado — default é evento raro, esperamos algo em torno de 5–10%.

In [ ]:
default_rate = df['SeriousDlqin2yrs'].mean()
print(f'Taxa de inadimplência: {default_rate:.4f} ({default_rate*100:.2f}%)')
print(df['SeriousDlqin2yrs'].value_counts())

## 4. Violações de domínio

**Hipótese:** Existem valores biologicamente/financeiramente impossíveis:
- `age < 18` ou `age > 100` — menor de idade ou biologicamente improvável
- `MonthlyIncome < 0` — renda não pode ser negativa
- `NumberOfDependents > 20` — outlier extremo

Cada condição usa máscara booleana separada — evita mascarar a origem do problema.

In [ ]:
print('Idades inválidas (<18 ou >100):', ((df['age'] < 18) | (df['age'] > 100)).sum())
print('Renda negativa:', (df['MonthlyIncome'] < 0).sum())
print('Dependentes > 20:', (df['NumberOfDependents'] > 20).sum())
print('DebtRatio > 100:', (df['DebtRatio'] > 100).sum())
print('RevolvingUtilization > 1.5:', (df['RevolvingUtilizationOfUnsecuredLines'] > 1.5).sum())

## 5. Distribuição de renda

**Hipótese:** A renda tem cauda direita pesada — P99 muito acima do P75.  
Se P99 >> P50, a **média** seria puxada pelos outliers e superestimaria o valor típico.  
**Conclusão:** usar **mediana** na imputação de `MonthlyIncome` em `merge_sources()`.

In [ ]:
percentis = df['MonthlyIncome'].quantile([0.25, 0.5, 0.75, 0.95, 0.99])
print('Percentis de MonthlyIncome:')
print(percentis.round(0))
print(f'\nMédia: {df["MonthlyIncome"].mean():.0f}')
print(f'Mediana (P50): {df["MonthlyIncome"].median():.0f}')
print(f'\nDiferença média vs mediana: {df["MonthlyIncome"].mean() - df["MonthlyIncome"].median():.0f}')

## Sumário das decisões de design

| Achado | Regra em `ingestion.py` |
|---|---|
| `MonthlyIncome` ~20% nulos | threshold `is_valid` < 30% |
| `age` < 18 ou > 100 existem | `age_invalid` em `violations` |
| `DebtRatio` > 100 existem (24k!) | `debt_ratio_high` em `violations` |
| `RevolvingUtilization` > 1.5 existem | `utilization_high` em `violations` |
| P99 renda >> P50 renda | imputação com mediana, não média |
| `NumberOfDependents` ausente = sem dependentes | imputação com 0 |

**Resultado esperado de `validate_dataframe(df)` no dataset bruto:**
- `is_valid: False` (violations existem)
- `missing_pct`: MonthlyIncome ~20%, NumberOfDependents ~3%

Depois de `merge_sources()`: **zero nulos**, flags de ausência preservadas como features.